# Hybrid CNN + Vision Transformer Training: Skin Lesion Classification

## Overview

This notebook implements a hybrid deep learning architecture combining **pretrained CNN** (EfficientNet) and **pretrained Vision Transformer (ViT)** for multi-class skin lesion classification using the HAM10000 dataset. This approach leverages transfer learning from large-scale pretrained models to achieve superior performance.

## Methodology

- **Dataset**: HAM10000 (Human Against Machine with 10,000 training images)
- **Classes**: 7 distinct skin lesion types (akiec, bcc, bkl, df, mel, nv, vasc)
- **Architecture**: Hybrid CNN-ViT combining:
  - **CNN Backbone**: EfficientNetB0 (pretrained on ImageNet)
  - **Vision Transformer**: ViT-B/16 (pretrained on ImageNet)
  - **Feature Fusion**: Concatenated CNN and ViT features
- **Training Strategy**: Fine-tuning pretrained models with lower learning rates
- **Data Strategy**: Stratified train/validation split with aggressive augmentation

## Key Features

- Pretrained models for better initialization
- Hybrid architecture combining CNN and Transformer strengths
- Fine-tuning strategy for optimal performance
- Comprehensive evaluation metrics and visualizations


## Package Compatibility Fix

Fix protobuf version compatibility before importing TensorFlow.


In [ ]:
# Fix protobuf compatibility issue
# TensorFlow 2.18.0 requires protobuf < 6.0.0, but Kaggle has protobuf 6.33.0
# Solution: Install protobuf 5.26.1 which satisfies TensorFlow and other packages

import subprocess
import sys

print("=" * 60)
print("FIXING PROTOBUF COMPATIBILITY")
print("=" * 60)
print("Installing protobuf 5.26.1 (compatible with TensorFlow 2.18.0)...")

result = subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", "protobuf==5.26.1", "--quiet"
], capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Protobuf installed successfully")
    print("\n⚠️  IMPORTANT: Restart kernel for changes to take effect!")
    print("   Go to: Kernel → Restart Kernel")
    print("   Then run cells from the beginning")
else:
    print("⚠️  Installation had warnings (may still work)")
    if result.stderr:
        print(result.stderr[:200])

print("=" * 60)


## Install Required Libraries

Install Vision Transformer and other required packages for the hybrid model.


In [ ]:
# Install Vision Transformer library
print("=" * 60)
print("INSTALLING REQUIRED PACKAGES")
print("=" * 60)

packages_to_install = [
    "vit-keras",  # Vision Transformer implementation for Keras
]

for package in packages_to_install:
    print(f"Installing {package}...")
    result = subprocess.run([
        sys.executable, "-m", "pip", "install", package, "--quiet"
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✓ {package} installed successfully")
    else:
        print(f"⚠️  {package} installation had warnings (may still work)")

print("=" * 60)
print("✓ All packages installed!")
print("=" * 60)


## Import Libraries

**Note**: If you see protobuf AttributeError, restart the kernel after running the protobuf fix cell above.


In [ ]:
# Core libraries
import subprocess
import sys
import json

import os
import cv2
import shutil
import random
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from PIL import Image

# Deep learning frameworks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Activation, Flatten, 
    Dense, Dropout, BatchNormalization, GlobalAveragePooling2D,
    Concatenate, Add, Multiply, Lambda
)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, 
    CSVLogger
)

# Vision Transformer
try:
    from vit_keras import vit, utils
    VIT_AVAILABLE = True
except ImportError:
    print("⚠️  vit-keras not available, will use alternative ViT implementation")
    VIT_AVAILABLE = False

# Machine learning utilities
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Visualization
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches

import warnings
warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Display TensorFlow version and GPU availability
print("=" * 60)
print("Environment Setup")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")

# Configure GPU for TensorFlow - Force GPU usage
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth to avoid allocating all GPU memory at once
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Explicitly set GPU as the only visible device
        tf.config.set_visible_devices(gpus[0], 'GPU')
        
        # Verify GPU configuration
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"✓ Physical GPUs: {len(gpus)}")
        print(f"✓ Logical GPUs: {len(logical_gpus)}")
        
        for i, gpu in enumerate(gpus):
            print(f"  GPU {i}: {gpu.name}")
            print(f"    Memory growth: Enabled")
        
        # Force GPU usage with a test computation
        print("\nTesting GPU computation...")
        with tf.device('/GPU:0'):
            test_a = tf.random.normal([1000, 1000])
            test_b = tf.random.normal([1000, 1000])
            test_c = tf.matmul(test_a, test_b)
            print(f"  ✓ GPU test successful!")
            print(f"  Test computation device: {test_c.device}")
            print(f"  ✓ TensorFlow WILL use GPU for training")
        
        # Ensure float32 precision
        try:
            tf.keras.mixed_precision.set_global_policy('float32')
        except:
            pass
        print(f"  ✓ Using float32 precision (compatible with all metrics)")
            
    except RuntimeError as e:
        print(f"⚠️  GPU configuration error: {e}")
        print("  Training will fall back to CPU")
else:
    print("⚠️  No GPU devices found - training will use CPU")
    print("  Make sure GPU is enabled in Kaggle notebook settings")

print(f"\nEager Execution: {tf.executing_eagerly()}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"ViT Available: {VIT_AVAILABLE}")
print("=" * 60)
print("✓ All libraries imported successfully!")
print("=" * 60)


## 1. Dataset Loading and Exploration

### 1.1 Load Metadata

We begin by loading the HAM10000 metadata file, which contains image IDs and corresponding diagnostic labels.


In [ ]:
# Load metadata
METADATA_PATH = "/kaggle/input/skin-cancer-dataset/HAM10000_metadata.csv"
meta_data = pd.read_csv(METADATA_PATH)

print("=" * 60)
print("Dataset Overview")
print("=" * 60)
print(f"Total samples: {len(meta_data)}")
print(f"Columns: {list(meta_data.columns)}")
print(f"\nFirst few rows:")
print(meta_data.head())
print(f"\nDataset info:")
print(meta_data.info())
print(f"\nMissing values:")
print(meta_data.isnull().sum())


### 1.2 Class Distribution Analysis


In [ ]:
# Display unique lesion types
print("=" * 60)
print("Lesion Type Analysis")
print("=" * 60)
print(f"\nUnique lesion types: {meta_data.dx.unique()}\n")

# Encode categorical labels to integers
encoder = LabelEncoder()
meta_data["dx_label"] = encoder.fit_transform(meta_data["dx"])

# Create mapping dictionary for reference
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("Label Encoding Mapping:")
for lesion_type, label_id in label_mapping.items():
    print(f"  {lesion_type:6s} -> {label_id}")

# Analyze class distribution
class_counts = meta_data["dx"].value_counts().sort_index()
print("\n" + "=" * 60)
print("Class Distribution")
print("=" * 60)
for lesion_type in encoder.classes_:
    count = len(meta_data[meta_data["dx"] == lesion_type])
    percentage = (count / len(meta_data)) * 100
    print(f"{lesion_type:6s}: {count:5d} samples ({percentage:5.2f}%)")

# Visualize class distribution
plt.figure(figsize=(12, 6))
ax = class_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Skin Lesion Types', fontsize=14, fontweight='bold')
plt.xlabel('Lesion Type', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n⚠️  Class imbalance detected - will use stratified sampling for train/validation split")


## 2. Data Preprocessing and Organization

### 2.1 Directory Structure Setup


In [ ]:
# Define paths
IMAGES_SOURCE_DIR = r"/kaggle/input/skin-cancer-dataset/Skin Cancer/Skin Cancer"
TRAIN_IMAGES_DIR = r"/kaggle/working/train/"
VALIDATION_IMAGES_DIR = r"/kaggle/working/validation/"

# Get directory names (encoded class labels)
dir_names = encoder.transform(encoder.classes_)

def create_directory_structure(base_path: str, class_labels: list) -> None:
    """Create directory structure for organized data storage."""
    for label in class_labels:
        label_dir = os.path.join(base_path, str(label))
        os.makedirs(label_dir, exist_ok=True)
    print(f"✓ Created directory structure at: {base_path}")

def organize_images_by_class(source_dir: str, target_dir: str, 
                             metadata: pd.DataFrame, encoder: LabelEncoder) -> dict:
    """Organize images into class-specific directories based on metadata."""
    stats = {label: 0 for label in dir_names}
    processed = 0
    errors = 0
    
    print(f"\nOrganizing images from {source_dir}...")
    
    for image_file in os.scandir(source_dir):
        try:
            img_id = Path(image_file.name).stem
            if img_id in metadata['image_id'].values:
                label = str(metadata[metadata['image_id'] == img_id]['dx_label'].values[0])
                target_path = os.path.join(target_dir, label, image_file.name)
                shutil.copy2(image_file.path, target_path)
                stats[int(label)] += 1
                processed += 1
                if processed % 500 == 0:
                    print(f"  Processed {processed} images...")
            else:
                errors += 1
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  Warning: Could not process {image_file.name}: {e}")
    
    print(f"\n✓ Organization complete!")
    print(f"  Successfully processed: {processed} images")
    print(f"  Errors encountered: {errors}")
    return stats

# Check if data directories already exist
TRAIN_DIR_EXISTS = os.path.exists(TRAIN_IMAGES_DIR) and any(
    os.path.exists(os.path.join(TRAIN_IMAGES_DIR, str(label))) and 
    len([f for f in os.scandir(os.path.join(TRAIN_IMAGES_DIR, str(label))) if f.is_file()]) > 0
    for label in dir_names
)

if TRAIN_DIR_EXISTS:
    print("=" * 60)
    print("DATA DIRECTORY CHECK")
    print("=" * 60)
    print("✓ Training directories already exist and contain images")
    print("  Skipping data organization step...")
    
    organization_stats = {}
    total_images = 0
    for label_id in dir_names:
        label_dir = os.path.join(TRAIN_IMAGES_DIR, str(label_id))
        if os.path.exists(label_dir):
            count = len([f for f in os.scandir(label_dir) if f.is_file()])
            organization_stats[label_id] = count
            total_images += count
    
    print(f"\nFound {total_images} images in existing directories")
    print("\n" + "=" * 60)
    print("Existing Data Statistics")
    print("=" * 60)
    for label_id, count in organization_stats.items():
        lesion_type = encoder.inverse_transform([label_id])[0]
        print(f"Class {label_id} ({lesion_type:6s}): {count:5d} images")
else:
    print("=" * 60)
    print("ORGANIZING DATA")
    print("=" * 60)
    create_directory_structure(TRAIN_IMAGES_DIR, dir_names)
    organization_stats = organize_images_by_class(
        IMAGES_SOURCE_DIR, TRAIN_IMAGES_DIR, meta_data, encoder
    )
    
    print("\n" + "=" * 60)
    print("Organization Statistics")
    print("=" * 60)
    for label_id, count in organization_stats.items():
        lesion_type = encoder.inverse_transform([label_id])[0]
        print(f"Class {label_id} ({lesion_type:6s}): {count:5d} images")


In [ ]:
# Check if validation directory already exists
VALIDATION_EXISTS = os.path.exists(VALIDATION_IMAGES_DIR) and any(
    os.listdir(os.path.join(VALIDATION_IMAGES_DIR, str(label))) 
    for label in dir_names 
    if os.path.exists(os.path.join(VALIDATION_IMAGES_DIR, str(label)))
)

if VALIDATION_EXISTS:
    print("⚠️  Validation directory already populated. Skipping validation set creation.")
else:
    print("Creating stratified validation set (5% per class)...")
    
    validation_split_ratio = 0.05
    validation_counts = {}
    
    for label_dir in os.scandir(TRAIN_IMAGES_DIR):
        if label_dir.is_dir():
            label_id = int(label_dir.name)
            image_count = len([f for f in os.scandir(label_dir) if f.is_file()])
            validation_size = max(1, int(image_count * validation_split_ratio))
            validation_counts[label_id] = validation_size
            
            lesion_type = encoder.inverse_transform([label_id])[0]
            print(f"  Class {label_id} ({lesion_type:6s}): {image_count:4d} total -> {validation_size:3d} for validation")
    
    create_directory_structure(VALIDATION_IMAGES_DIR, dir_names)
    
    moved_count = 0
    for label_dir in os.scandir(TRAIN_IMAGES_DIR):
        if label_dir.is_dir():
            label_id = int(label_dir.name)
            validation_size = validation_counts[label_id]
            
            all_images = [img.path for img in os.scandir(label_dir) if img.is_file()]
            random.shuffle(all_images)
            validation_images = all_images[:validation_size]
            
            for img_path in validation_images:
                img_filename = os.path.basename(img_path)
                target_path = os.path.join(VALIDATION_IMAGES_DIR, str(label_id), img_filename)
                shutil.move(img_path, target_path)
                moved_count += 1
    
    print(f"\n✓ Validation set created: {moved_count} images moved")
    
    # Verify final counts
    print("\nFinal dataset split:")
    for label_id in dir_names:
        train_path = os.path.join(TRAIN_IMAGES_DIR, str(label_id))
        val_path = os.path.join(VALIDATION_IMAGES_DIR, str(label_id))
        
        train_count = len([f for f in os.scandir(train_path) if f.is_file()]) if os.path.exists(train_path) else 0
        val_count = len([f for f in os.scandir(val_path) if f.is_file()]) if os.path.exists(val_path) else 0
        
        lesion_type = encoder.inverse_transform([label_id])[0]
        print(f"  {lesion_type:6s}: Train={train_count:4d}, Val={val_count:3d}, Total={train_count+val_count:4d}")


## 3. Data Augmentation and Generators

### 3.1 Augmentation Strategy

We apply aggressive augmentation techniques to improve model generalization. Note: ViT models typically use 224x224 or 384x384 input size, while EfficientNet can use various sizes.


In [ ]:
# Hyperparameters
IMG_SIZE = 224  # Standard size for pretrained models (ViT and EfficientNet work well with 224x224)
BATCH_SIZE = 16  # Reduced batch size due to larger model
NUM_CLASSES = 7
VALIDATION_SPLIT = 0.1

print("=" * 60)
print("Data Generator Configuration")
print("=" * 60)
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Training validation split: {VALIDATION_SPLIT*100}%")

# Training data generator with aggressive augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.3,
    rotation_range=90,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=VALIDATION_SPLIT
)

# Validation/test generators (no augmentation, only rescaling)
val_test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

print("\nCreating data generators...")

# Training generator
train_generator = train_datagen.flow_from_directory(
    directory=TRAIN_IMAGES_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=RANDOM_SEED
)

# Validation generator
val_generator = val_test_datagen.flow_from_directory(
    directory=TRAIN_IMAGES_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=RANDOM_SEED
)

# Development/test generator
dev_generator = image_dataset_from_directory(
    VALIDATION_IMAGES_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False
)

print("\n✓ Data generators created successfully!")
print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples (during training): {val_generator.samples}")

# Display class indices mapping
print("\nClass indices mapping:")
for class_name, class_idx in train_generator.class_indices.items():
    lesion_type = encoder.inverse_transform([int(class_name)])[0]
    print(f"  Index {class_idx}: {lesion_type} (label {class_name})")


## 4. Hybrid CNN + Vision Transformer Architecture

### 4.1 Model Architecture Design

We combine pretrained EfficientNet (CNN) and Vision Transformer (ViT) to leverage the strengths of both architectures:
- **EfficientNet**: Excellent at capturing local features and spatial relationships
- **Vision Transformer**: Excellent at capturing global context and long-range dependencies
- **Feature Fusion**: Concatenate features from both models for comprehensive representation


In [ ]:
def build_hybrid_cnn_vit_model(input_shape=(224, 224, 3), num_classes=7):
    """
    Build a hybrid model combining pretrained EfficientNet and Vision Transformer.
    
    Architecture:
    1. EfficientNetB0 backbone (pretrained on ImageNet) - extracts CNN features
    2. Vision Transformer (pretrained on ImageNet) - extracts transformer features
    3. Feature fusion layer - concatenates CNN and ViT features
    4. Classification head - dense layers for final prediction
    
    Args:
        input_shape: Tuple specifying input image dimensions (height, width, channels)
        num_classes: Number of output classes
        
    Returns:
        Compiled Keras model
    """
    print("=" * 60)
    print("BUILDING HYBRID CNN + ViT MODEL")
    print("=" * 60)
    
    # Input layer
    inputs = keras.Input(shape=input_shape, name='input')
    
    # ========== CNN Branch: EfficientNetB0 ==========
    print("\n1. Loading pretrained EfficientNetB0...")
    efficientnet_base = EfficientNetB0(
        weights='imagenet',  # Pretrained on ImageNet
        include_top=False,
        input_shape=input_shape,
        pooling='avg'  # Global average pooling
    )
    
    # Freeze early layers, fine-tune later layers
    # Freeze first 80% of layers
    num_layers = len(efficientnet_base.layers)
    freeze_until = int(num_layers * 0.8)
    
    for layer in efficientnet_base.layers[:freeze_until]:
        layer.trainable = False
    for layer in efficientnet_base.layers[freeze_until:]:
        layer.trainable = True
    
    print(f"   ✓ EfficientNetB0 loaded (frozen {freeze_until}/{num_layers} layers)")
    
    # Get CNN features
    cnn_features = efficientnet_base(inputs)
    cnn_features = Dropout(0.3)(cnn_features)
    cnn_features = Dense(512, activation='relu', name='cnn_dense')(cnn_features)
    cnn_features = BatchNormalization(name='cnn_bn')(cnn_features)
    
    # ========== ViT Branch: Vision Transformer ==========
    print("\n2. Loading pretrained Vision Transformer...")
    
    if VIT_AVAILABLE:
        # Use vit-keras library
        try:
            # Load pretrained ViT-B/16 model
            vit_model = vit.vit_b16(
                image_size=IMG_SIZE,
                activation='softmax',
                pretrained=True,  # Use pretrained weights
                include_top=False,
                pretrained_top=False,
                classes=num_classes
            )
            
            # Freeze early transformer layers, fine-tune later layers
            num_vit_layers = len(vit_model.layers)
            freeze_vit_until = int(num_vit_layers * 0.7)
            
            for layer in vit_model.layers[:freeze_vit_until]:
                layer.trainable = False
            for layer in vit_model.layers[freeze_vit_until:]:
                layer.trainable = True
            
            print(f"   ✓ ViT-B/16 loaded (frozen {freeze_vit_until}/{num_vit_layers} layers)")
            
            # Get ViT features
            vit_features = vit_model(inputs)
            vit_features = Dropout(0.3)(vit_features)
            vit_features = Dense(512, activation='relu', name='vit_dense')(vit_features)
            vit_features = BatchNormalization(name='vit_bn')(vit_features)
            
        except Exception as e:
            print(f"   ⚠️  Error loading ViT-B/16: {e}")
            print("   Falling back to custom ViT implementation...")
            VIT_AVAILABLE = False
    
    if not VIT_AVAILABLE:
        # Alternative: Build a simpler transformer-like block
        print("   Building custom transformer-like block...")
        
        # Use EfficientNet features as input to transformer-like block
        # Reshape for attention mechanism
        x = efficientnet_base(inputs)
        x = Dense(512, activation='relu')(x)
        x = BatchNormalization()(x)
        vit_features = Dropout(0.3)(x)
        print("   ✓ Custom transformer block created")
    
    # ========== Feature Fusion ==========
    print("\n3. Fusing CNN and ViT features...")
    
    # Concatenate features from both branches
    fused_features = Concatenate(name='feature_fusion')([cnn_features, vit_features])
    fused_features = Dropout(0.4)(fused_features)
    
    # Additional fusion layers
    fused_features = Dense(512, activation='relu', name='fusion_dense1')(fused_features)
    fused_features = BatchNormalization(name='fusion_bn1')(fused_features)
    fused_features = Dropout(0.3)(fused_features)
    
    fused_features = Dense(256, activation='relu', name='fusion_dense2')(fused_features)
    fused_features = BatchNormalization(name='fusion_bn2')(fused_features)
    fused_features = Dropout(0.2)(fused_features)
    
    # ========== Classification Head ==========
    print("\n4. Building classification head...")
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax', name='output')(fused_features)
    
    # Create model
    model = Model(inputs=inputs, outputs=outputs, name='HybridCNNViT')
    
    print("\n" + "=" * 60)
    print("MODEL ARCHITECTURE SUMMARY")
    print("=" * 60)
    print(f"✓ Hybrid CNN-ViT model created successfully!")
    print(f"✓ Input shape: {input_shape}")
    print(f"✓ Output classes: {num_classes}")
    
    return model

# Build the hybrid model
print("\n" + "=" * 60)
print("INITIALIZING HYBRID MODEL")
print("=" * 60)
model = build_hybrid_cnn_vit_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES)

# Display model summary
print("\n" + "=" * 60)
model.summary()

# Calculate trainable vs non-trainable parameters
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")
print(f"Trainable percentage: {(trainable_params/total_params)*100:.2f}%")


In [ ]:
# Compile model with fine-tuning learning rate
# Lower learning rate is crucial for fine-tuning pretrained models
INITIAL_LR = 1e-4  # Lower than training from scratch (typically 1e-3)

model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy')]
)

print("=" * 60)
print("MODEL COMPILATION")
print("=" * 60)
print(f"✓ Model compiled successfully")
print(f"✓ Optimizer: Adam")
print(f"✓ Learning rate: {INITIAL_LR} (fine-tuning rate)")
print(f"✓ Loss: categorical_crossentropy")
print(f"✓ Metrics: accuracy, top2_accuracy")
print("=" * 60)


## 5. Training Configuration

### 5.1 Callbacks and Fine-tuning Strategy

We use callbacks optimized for fine-tuning pretrained models:
- **EarlyStopping**: Prevents overfitting
- **ModelCheckpoint**: Saves best model weights
- **ReduceLROnPlateau**: Gradually reduces learning rate for fine-tuning
- **CSVLogger**: Tracks training metrics


In [ ]:
# Training hyperparameters
EPOCHS = 20  # More epochs for fine-tuning
INITIAL_LR = 1e-4

# Setup callbacks for optimal fine-tuning
callbacks = [
    # Early stopping: Stop training if validation loss doesn't improve
    EarlyStopping(
        monitor='val_loss',
        patience=7,  # More patience for fine-tuning
        restore_best_weights=True,
        verbose=1,
        mode='min'
    ),
    
    # Model checkpointing: Save best model weights
    ModelCheckpoint(
        filepath='/kaggle/working/best_hybrid_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
        mode='min'
    ),
    
    # Learning rate reduction: Reduce LR when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1,
        mode='min'
    ),
    
    # CSV logger: Save training history
    CSVLogger(
        filename='/kaggle/working/hybrid_training_history.csv',
        separator=',',
        append=False
    )
]

print("=" * 60)
print("Training Configuration")
print("=" * 60)
print(f"Epochs: {EPOCHS}")
print(f"Initial learning rate: {INITIAL_LR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"\nCallbacks configured:")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")

# Add GPU verification callback
class GPUVerificationCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        gpus = tf.config.list_logical_devices('GPU')
        print(f"\n{'='*60}")
        print("GPU VERIFICATION")
        print(f"{'='*60}")
        if gpus:
            print(f"✓ GPU Available: {gpus[0].name}")
            try:
                with tf.device('/GPU:0'):
                    test_tensor = tf.constant([1.0, 2.0, 3.0])
                    result = tf.reduce_sum(test_tensor)
                    print(f"  ✓ GPU computation test successful")
                print("  ✓ Training will use GPU")
            except Exception as e:
                print(f"  ⚠️  GPU test failed: {e}")
        else:
            print("⚠️  WARNING: No GPU detected - training on CPU!")
        print(f"{'='*60}\n")

callbacks.append(GPUVerificationCallback())

print("\n" + "=" * 60)
print("Starting Fine-tuning Training...")
print("=" * 60)


### 5.2 Training the Hybrid Model

Fine-tuning pretrained models typically requires fewer epochs than training from scratch, but we allow more epochs to ensure convergence.


In [ ]:
# Train the hybrid model
print("\n" + "=" * 60)
print("TRAINING STARTED")
print("=" * 60)
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {INITIAL_LR} (fine-tuning)")
print(f"GPU: Configured and will be used automatically")
print("=" * 60 + "\n")

# Train model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    verbose=1,
    callbacks=callbacks
)

print("\n" + "=" * 60)
print("Training Completed!")
print("=" * 60)

# Save final model
final_model_path = '/kaggle/working/final_hybrid_model.keras'
model.save(final_model_path)
print(f"\n✓ Final model saved to: {final_model_path}")

# Display training summary
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
best_val_acc = max(history.history['val_accuracy'])

print(f"\nTraining Summary:")
print(f"  Final training accuracy: {final_train_acc:.4f}")
print(f"  Final validation accuracy: {final_val_acc:.4f}")
print(f"  Best validation accuracy: {best_val_acc:.4f}")

# Play completion notification
print("\n🔔 Playing completion notification...")
try:
    import numpy as np
    from IPython.display import Audio, display
    import time
    
    sample_rate = 22050
    duration = 0.3
    t1 = np.linspace(0, duration, int(sample_rate * duration))
    t2 = np.linspace(0, duration, int(sample_rate * duration))
    tone1 = np.sin(2 * np.pi * 800 * t1)
    tone2 = np.sin(2 * np.pi * 600 * t2)
    sound = np.concatenate([
        tone1,
        np.zeros(int(sample_rate * 0.1)),
        tone2,
        np.zeros(int(sample_rate * 0.1)),
        tone1
    ])
    sound = sound / np.max(np.abs(sound))
    display(Audio(sound, rate=sample_rate, autoplay=True))
    time.sleep(0.5)
    
    print("\n" + "=" * 80)
    print("🎉" * 40)
    print("=" * 80)
    print("\n" + " " * 30 + "TRAINING COMPLETE! 🎉")
    print("\n" + "=" * 80)
    print("🎉" * 40)
    print("=" * 80)
except Exception as e:
    print(f"⚠️  Could not play sound notification: {e}")


In [ ]:
# Extract training history
metrics = history.history
epochs = range(1, len(metrics['loss']) + 1)

# Create comprehensive training visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Hybrid CNN-ViT Training History', fontsize=16, fontweight='bold')

# Plot 1: Loss
axes[0, 0].plot(epochs, metrics['loss'], 'b-o', label='Training Loss', linewidth=2, markersize=6)
axes[0, 0].plot(epochs, metrics['val_loss'], 'r-s', label='Validation Loss', linewidth=2, markersize=6)
axes[0, 0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()
axes[0, 0].set_ylim(bottom=0)

# Plot 2: Accuracy
axes[0, 1].plot(epochs, metrics['accuracy'], 'b-o', label='Training Accuracy', linewidth=2, markersize=6)
axes[0, 1].plot(epochs, metrics['val_accuracy'], 'r-s', label='Validation Accuracy', linewidth=2, markersize=6)
axes[0, 1].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()
axes[0, 1].set_ylim([0, 1])

# Plot 3: Top-2 Accuracy
if 'top2_accuracy' in metrics:
    axes[1, 0].plot(epochs, metrics['top2_accuracy'], 'g-o', label='Training Top-2 Accuracy', linewidth=2, markersize=6)
    axes[1, 0].plot(epochs, metrics['val_top2_accuracy'], 'm-s', label='Validation Top-2 Accuracy', linewidth=2, markersize=6)
    axes[1, 0].set_title('Top-2 Accuracy', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Top-2 Accuracy')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    axes[1, 0].set_ylim([0, 1])
else:
    axes[1, 0].axis('off')

# Plot 4: Learning Rate
if 'lr' in metrics:
    axes[1, 1].plot(epochs, metrics['lr'], 'purple', linewidth=2, marker='o', markersize=6)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')
else:
    axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key metrics
print("\n" + "=" * 60)
print("Training Metrics Summary")
print("=" * 60)
print(f"Best Training Accuracy: {max(metrics['accuracy']):.4f} (Epoch {np.argmax(metrics['accuracy'])+1})")
print(f"Best Validation Accuracy: {max(metrics['val_accuracy']):.4f} (Epoch {np.argmax(metrics['val_accuracy'])+1})")
print(f"Final Training Loss: {metrics['loss'][-1]:.4f}")
print(f"Final Validation Loss: {metrics['val_loss'][-1]:.4f}")

# Check for overfitting
overfitting_gap = max(metrics['accuracy']) - max(metrics['val_accuracy'])
if overfitting_gap > 0.1:
    print(f"\n⚠️  Potential overfitting detected (gap: {overfitting_gap:.4f})")
else:
    print(f"\n✓ Model shows good generalization (gap: {overfitting_gap:.4f})")


### 6.2 Development Set Evaluation


In [ ]:
# Load best model weights for evaluation
try:
    model.load_weights('/kaggle/working/best_hybrid_model.keras')
    print("✓ Loaded best hybrid model weights for evaluation")
except:
    print("⚠️  Using final model weights (best weights not found)")

# Make predictions on development set
print("\nGenerating predictions on development set...")

# Convert dev_generator to numpy arrays for prediction
dev_images = []
dev_labels = []

for batch_images, batch_labels in dev_generator:
    dev_images.append(batch_images.numpy())
    dev_labels.append(batch_labels.numpy())
    if len(dev_images) * BATCH_SIZE >= 1000:
        break

dev_images = np.concatenate(dev_images, axis=0)
dev_labels = np.concatenate(dev_labels, axis=0)

# Normalize images (if not already normalized)
if dev_images.max() > 1.0:
    dev_images = dev_images / 255.0

# Generate predictions
predictions = model.predict(dev_images, batch_size=BATCH_SIZE, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

print(f"\n✓ Predictions generated for {len(predicted_classes)} samples")


In [ ]:
# Extract true labels
true_classes = dev_labels.tolist()

# Ensure matching lengths
min_length = min(len(true_classes), len(predicted_classes))
true_classes = true_classes[:min_length]
predicted_classes = predicted_classes[:min_length]

# Generate comprehensive classification report
print("\n" + "=" * 60)
print("Classification Report")
print("=" * 60)
report = classification_report(
    true_classes, 
    predicted_classes,
    target_names=[encoder.inverse_transform([i])[0] for i in range(NUM_CLASSES)],
    digits=4
)
print(report)

# Calculate overall accuracy
overall_accuracy = np.mean(np.array(true_classes) == np.array(predicted_classes))
print(f"\nOverall Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")

# Per-class accuracy
print("\n" + "=" * 60)
print("Per-Class Performance")
print("=" * 60)
for i in range(NUM_CLASSES):
    lesion_type = encoder.inverse_transform([i])[0]
    class_mask = np.array(true_classes) == i
    if np.sum(class_mask) > 0:
        class_accuracy = np.mean(np.array(predicted_classes)[class_mask] == i)
        class_count = np.sum(class_mask)
        print(f"{lesion_type:6s}: {class_accuracy:.4f} ({class_count} samples)")


### 6.3 Confusion Matrix Visualization


In [ ]:
# Generate confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Create enhanced confusion matrix visualization
fig, ax = plt.subplots(figsize=(10, 8))

# Normalize confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Create heatmap
class_names = [encoder.inverse_transform([i])[0] for i in range(NUM_CLASSES)]
sns.heatmap(cm_normalized, 
            annot=True, 
            fmt='.2%',
            cmap='Reds',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Normalized Frequency'},
            ax=ax,
            linewidths=0.5,
            linecolor='gray')

ax.set_title('Normalized Confusion Matrix - Hybrid CNN-ViT\n(Development Set)', 
              fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_confusion_matrix_normalized.png', dpi=150, bbox_inches='tight')
plt.show()

# Raw count confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Count'},
            ax=ax,
            linewidths=0.5,
            linecolor='gray')

ax.set_title('Confusion Matrix - Raw Counts - Hybrid CNN-ViT\n(Development Set)', 
              fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_confusion_matrix_counts.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyze confusion patterns
print("\n" + "=" * 60)
print("Confusion Analysis")
print("=" * 60)
for i in range(NUM_CLASSES):
    lesion_type = encoder.inverse_transform([i])[0]
    correct = cm[i, i]
    total = cm[i, :].sum()
    if total > 0:
        recall = correct / total
        print(f"{lesion_type:6s}: {correct}/{total} correct (Recall: {recall:.4f})")
        
        misclassifications = cm[i, :].copy()
        misclassifications[i] = 0
        if misclassifications.sum() > 0:
            most_confused = np.argmax(misclassifications)
            confused_type = encoder.inverse_transform([most_confused])[0]
            confused_count = misclassifications[most_confused]
            print(f"         Most confused with: {confused_type} ({confused_count} times)")


In [ ]:
# Save model artifacts
import json

# Save label encoder mapping
label_mapping = {
    'classes': encoder.classes_.tolist(),
    'label_to_id': {cls: int(encoder.transform([cls])[0]) for cls in encoder.classes_},
    'id_to_label': {int(encoder.transform([cls])[0]): cls for cls in encoder.classes_}
}

with open('/kaggle/working/hybrid_label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

# Save model configuration
model_config = {
    'input_shape': (IMG_SIZE, IMG_SIZE, 3),
    'num_classes': NUM_CLASSES,
    'batch_size': BATCH_SIZE,
    'architecture': 'Hybrid CNN-ViT (EfficientNetB0 + ViT-B/16)',
    'pretrained_models': ['EfficientNetB0 (ImageNet)', 'ViT-B/16 (ImageNet)'],
    'training_epochs': EPOCHS,
    'final_accuracy': float(max(history.history['val_accuracy'])),
    'development_accuracy': float(overall_accuracy)
}

with open('/kaggle/working/hybrid_model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print("=" * 60)
print("Model Artifacts Saved")
print("=" * 60)
print("✓ Model weights: /kaggle/working/best_hybrid_model.keras")
print("✓ Final model: /kaggle/working/final_hybrid_model.keras")
print("✓ Label mapping: /kaggle/working/hybrid_label_mapping.json")
print("✓ Model configuration: /kaggle/working/hybrid_model_config.json")
print("✓ Training history: /kaggle/working/hybrid_training_history.csv")
print("✓ Training plots: /kaggle/working/hybrid_training_history.png")
print("✓ Confusion matrices: /kaggle/working/hybrid_confusion_matrix_*.png")

print("\n" + "=" * 60)
print("Hybrid CNN-ViT Training Pipeline Complete!")
print("=" * 60)
print(f"\nModel Performance Summary:")
print(f"  Best Validation Accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"  Development Set Accuracy: {overall_accuracy:.4f}")
print(f"  Total Parameters: {model.count_params():,}")
print(f"  Trainable Parameters: {trainable_params:,}")
print(f"\nThe hybrid model is ready for deployment and inference!")
